## Sesión 2: IA tradicional
### Entrenamiento de un modelo predictivo para partidos de la Copa del Mundo

En esta sesión vamos a construir, paso a paso, un flujo clásico de **IA tradicional** aplicado a datos históricos de fútbol. La idea es que puedas ver cómo se prepara la información, cómo se crean variables útiles y cómo se entrena un modelo para estimar resultados.

Durante el notebook aprenderás a:
- Cargar datos desde la capa Bronze en AIDP.
- Limpiar y preparar los datos para análisis.
- Calcular una métrica ELO para representar la fuerza relativa de los equipos.
- Crear variables de entrada para modelos de machine learning.
- Entrenar modelos para predecir resultado y goles.
- Evaluar el comportamiento del modelo con datos de la Copa del Mundo 2022.
- Usar el modelo entrenado para simular partidos entre dos equipos.

> Recomendación para la sesión: ejecuta las celdas en orden y revisa la salida de cada bloque antes de continuar. Así será más fácil conectar cada etapa del proceso.

---

### Acceso a los datos

Comenzamos leyendo los datos cargados en la sesión anterior. Estos datos están en la capa **Bronze**, que normalmente contiene información cruda o con muy poco procesamiento.

En AIDP pueden existir diferentes esquemas según el ambiente o el usuario. Por eso, la siguiente celda intenta ubicar la tabla en más de un esquema y selecciona automáticamente el primero que esté disponible.


In [ ]:
schema_ok = None

for schema in ["admin", "oraclelabs"]:
    try:
        bronze_df = spark.table(f"deepdivecatalog_bronze.{schema}.bronze_wc_matches")
        schema_ok = schema
        break
    except Exception:
        pass

if schema_ok is None:
    raise Exception("No se encontró la tabla en los esquemas admin ni oraclelabs")

print(f"Esquema usado: {schema_ok}")
display(bronze_df.limit(5))


### Carga, limpieza y creación de la capa Silver

En esta etapa convertimos los datos crudos en una versión más limpia y consistente, lista para análisis y modelado.

El objetivo es realizar tareas básicas pero importantes:
- Estandarizar nombres de columnas.
- Reemplazar valores vacíos por valores nulos.
- Convertir la fecha del partido a un tipo de dato adecuado.
- Eliminar duplicados.
- Filtrar registros incompletos o inválidos.
- Convertir los marcadores a valores numéricos.

Al finalizar, guardaremos el resultado en una tabla Delta de la capa **Silver**. Esta capa representa datos más confiables para construir el modelo.


In [ ]:
from pyspark.sql.functions import col, to_date, when

normalized_columns = [c.lower().strip() for c in bronze_df.columns]

silver_df = (
    bronze_df
    
    # Estandarizar nombres de columnas para evitar diferencias por mayúsculas o espacios.
    .toDF(*normalized_columns)
    
    # Convertir cadenas vacías en valores nulos para facilitar filtros y validaciones.
    .select([
        when(col(c) == "", None).otherwise(col(c)).alias(c)
        for c in normalized_columns
    ])
    
    # Convertir la fecha del partido a un tipo de dato de fecha.
    .withColumn("match_date", to_date(col("match_date"), "M/d/yyyy"))
    
    # Eliminar registros duplicados exactos.
    .dropDuplicates()
    
    # Conservar solo partidos con ambos equipos informados.
    .filter(col("home_team_name").isNotNull())
    .filter(col("away_team_name").isNotNull())
)

silver_df = silver_df \
    .withColumn("home_team_score", col("home_team_score").cast("int")) \
    .withColumn("away_team_score", col("away_team_score").cast("int")) \
    .filter(col("home_team_score").isNotNull()) \
    .filter(col("away_team_score").isNotNull())


In [ ]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("deepdivecatalog_prata.silver_wc_matches")

### Preparación para el modelo: creación de variables

Los modelos de IA tradicional no aprenden directamente de una explicación en lenguaje natural; necesitan variables numéricas bien preparadas. A este proceso se le conoce como **feature engineering** o creación de variables.

En esta sección vamos a transformar los datos de partidos en señales útiles para el modelo, por ejemplo:
- Diferencia de fuerza entre equipos.
- Promedio de goles anotados.
- Promedio de goles recibidos.
- Porcentaje histórico de victorias.

#### Conversión a pandas

Para las siguientes transformaciones usaremos pandas, porque facilita el cálculo secuencial del ELO y la creación de métricas agregadas por equipo.


In [ ]:
df_pd = silver_df.orderBy("match_date").toPandas()

#### Cálculo de la variable ELO

El **ELO** es un sistema de puntuación usado para estimar la fuerza relativa de jugadores o equipos a partir de sus resultados históricos. En este notebook lo usaremos como una señal sencilla de desempeño acumulado.

La lógica general es:
- Todos los equipos comienzan con una puntuación inicial de 1500.
- Después de cada partido, la puntuación se ajusta según el resultado.
- Ganar contra un rival fuerte aporta más valor que ganar contra un rival débil.
- Perder reduce la puntuación, y empatar genera un ajuste intermedio.

Con esta variable, el modelo no solo observa el marcador de un partido, sino también una representación histórica de la fortaleza de cada equipo antes de jugar.

Al final de la celda se agregan tres columnas al conjunto de datos:
- `elo_home`: ELO del equipo local antes del partido.
- `elo_away`: ELO del equipo visitante antes del partido.
- `elo_diff`: diferencia entre ambos valores de ELO.


In [ ]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(rating, expected, actual, k=30):
    return rating + k * (actual - expected)


def compute_elo(df, k=30):

    df = df.sort_values("match_date").copy()

    ratings = {}
    elo_home = []
    elo_away = []

    for _, row in df.iterrows():

        home = row["home_team_name"]
        away = row["away_team_name"]

        # Cada equipo inicia con 1500 puntos si todavía no aparece en el histórico.
        ratings.setdefault(home, 1500)
        ratings.setdefault(away, 1500)

        rating_home = ratings[home]
        rating_away = ratings[away]

        elo_home.append(rating_home)
        elo_away.append(rating_away)

        expected_home = expected_score(rating_home, rating_away)
        expected_away = expected_score(rating_away, rating_home)

        if row["home_team_win"] == 1:
            actual_home, actual_away = 1, 0
        elif row["away_team_win"] == 1:
            actual_home, actual_away = 0, 1
        else:
            actual_home = actual_away = 0.5

        # Actualizar el ELO después del partido para que el siguiente registro use el historial acumulado.
        ratings[home] = update_elo(rating_home, expected_home, actual_home, k)
        ratings[away] = update_elo(rating_away, expected_away, actual_away, k)

    df["elo_home"] = elo_home
    df["elo_away"] = elo_away
    df["elo_diff"] = df["elo_home"] - df["elo_away"]

    return df, ratings


df, ratings = compute_elo(df_pd)


#### Creación de métricas históricas por equipo

Además del ELO, vamos a construir métricas simples que resumen el comportamiento histórico de cada equipo.

Estas métricas ayudan al modelo a comparar al equipo local y al visitante usando información como:
- Promedio de goles anotados.
- Promedio de goles recibidos.
- Porcentaje de partidos ganados.

Luego usaremos diferencias entre ambos equipos como variables de entrada. Por ejemplo, si un equipo suele anotar más que su rival, esa diferencia puede aportar información al modelo.


In [ ]:
import pandas as pd

def create_team_features(df):

    # Unificar partidos de local y visitante para calcular métricas por equipo.
    goals_for = pd.concat([
        df[["home_team_name", "home_team_score"]].rename(
            columns={"home_team_name": "Team", "home_team_score": "Goals_For"}
        ),
        df[["away_team_name", "away_team_score"]].rename(
            columns={"away_team_name": "Team", "away_team_score": "Goals_For"}
        )
    ])

    goals_against = pd.concat([
        df[["home_team_name", "away_team_score"]].rename(
            columns={"home_team_name": "Team", "away_team_score": "Goals_Against"}
        ),
        df[["away_team_name", "home_team_score"]].rename(
            columns={"away_team_name": "Team", "home_team_score": "Goals_Against"}
        )
    ])

    wins = pd.concat([
        df[["home_team_name", "home_team_win"]].rename(
            columns={"home_team_name": "Team", "home_team_win": "Win"}
        ),
        df[["away_team_name", "away_team_win"]].rename(
            columns={"away_team_name": "Team", "away_team_win": "Win"}
        )
    ])

    team_stats = goals_for.groupby("Team").mean()
    team_stats["avg_goals_against"] = goals_against.groupby("Team").mean()
    team_stats["win_rate"] = wins.groupby("Team").mean()

    return team_stats.fillna(0)


team_stats = create_team_features(df)


In [ ]:
def add_match_features(df, team_stats):

    df = df.copy()

    # Agregar métricas históricas del equipo local.
    df = df.merge(
        team_stats,
        left_on="home_team_name",
        right_index=True,
        how="left"
    ).rename(columns={
        "Goals_For": "teamA_avg_goals",
        "avg_goals_against": "teamA_avg_conceded",
        "win_rate": "teamA_win_rate"
    })

    # Agregar métricas históricas del equipo visitante.
    df = df.merge(
        team_stats,
        left_on="away_team_name",
        right_index=True,
        how="left",
        suffixes=("", "_B")
    ).rename(columns={
        "Goals_For": "teamB_avg_goals",
        "avg_goals_against": "teamB_avg_conceded",
        "win_rate": "teamB_win_rate"
    })

    df["goals_diff"] = df["teamA_avg_goals"] - df["teamB_avg_goals"]
    df["defense_diff"] = df["teamA_avg_conceded"] - df["teamB_avg_conceded"]
    df["win_rate_diff"] = df["teamA_win_rate"] - df["teamB_win_rate"]

    return df


df = add_match_features(df, team_stats)


#### Preparación del dataset de entrenamiento

Ahora definimos qué columnas serán usadas como variables de entrada del modelo y cuál será la variable objetivo.

En este caso, el modelo intentará clasificar el resultado del partido en tres categorías:
- `0`: empate.
- `1`: victoria del equipo local.
- `2`: victoria del equipo visitante.

También conservamos los goles de cada equipo para entrenar modelos adicionales que estimen el marcador esperado.


In [ ]:
def prepare_training_data(df: pd.DataFrame):

    features = [
        "elo_diff",
        "goals_diff",
        "defense_diff",
        "win_rate_diff"
    ]

    # Crear etiquetas: empate -> 0, victoria local -> 1, victoria visitante -> 2.
    df["match_result"] = (
        df["home_team_win"] * 1 +
        df["away_team_win"] * 2
    )

    df["match_result"] = pd.to_numeric(df["match_result"], errors="coerce")
    df = df.dropna(subset=["match_result"])
    df["match_result"] = df["match_result"].astype(int)

    X = df[features]
    y_result = df["match_result"]
    y_home_goals = df["home_team_score"]
    y_away_goals = df["away_team_score"]

    return X, y_result, y_home_goals, y_away_goals

X, y_result, y_home, y_away = prepare_training_data(df)


#### Entrenamiento de modelos de clasificación

En esta etapa entrenaremos modelos de machine learning para predecir el resultado del partido.

Primero dividimos los datos en dos grupos:
- **Entrenamiento**: datos que el modelo usa para aprender patrones.
- **Prueba**: datos que se reservan para evaluar si el modelo generaliza razonablemente bien.

Aunque una división común es 70/30, en este notebook usamos **80/20** para dejar más datos disponibles para entrenamiento, ya que el conjunto de datos es relativamente pequeño.

La siguiente celda instala `scikit-learn`, una biblioteca muy usada para modelos clásicos de machine learning. En algunos ambientes de AIDP puede estar instalada previamente; si ya existe, la instalación simplemente confirmará que está disponible.


In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y_result, test_size=0.2, random_state=42
)

models = {}

# Modelo 1: Random Forest, útil para capturar relaciones no lineales entre variables.
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

models["rf"] = rf_model

# Modelo 2: Regresión logística, una referencia simple para comparar desempeño.
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

models["logistic"] = log_model


#### Evaluación inicial del modelo

Después de entrenar los modelos, evaluamos su desempeño usando el 20% de datos que dejamos reservado para prueba.

La métrica que veremos es **accuracy** o exactitud: indica qué proporción de predicciones coincidió con el resultado real. Es una medida fácil de interpretar, aunque no debe ser la única métrica en proyectos reales.


In [ ]:
for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name}: exactitud = {acc:.4f}")


#### Modelo predictivo de goles

Además de clasificar el resultado del partido, entrenaremos dos modelos de regresión para estimar el número de goles esperados:
- Goles del equipo local.
- Goles del equipo visitante.

Esto nos permite mostrar una predicción más completa: probabilidades de resultado y marcador estimado.


In [ ]:
home_goals_model = RandomForestRegressor()
home_goals_model.fit(X, y_home)

away_goals_model = RandomForestRegressor()
away_goals_model.fit(X, y_away)

## Evaluación con datos de la Copa del Mundo 2022

Ahora probaremos el modelo con partidos de 2022. Esta validación nos ayuda a revisar cómo se comporta el modelo frente a un subconjunto específico y conocido del histórico.

Es importante recordar que este ejercicio tiene un propósito didáctico. En un proyecto productivo se revisarían más aspectos, como fuga de datos, validación temporal, balance de clases y ajuste de hiperparámetros.


In [ ]:
df["year"] = pd.to_datetime(df["match_date"]).dt.year
df_2022 = df[df["year"] == 2022]

X_2022 = df_2022[["elo_diff", "goals_diff", "defense_diff", "win_rate_diff"]]
y_2022 = df_2022["match_result"]

model = models["rf"]

preds_2022 = model.predict(X_2022)

print("Exactitud en partidos de la Copa 2022:", accuracy_score(y_2022, preds_2022))


### Matriz de confusión

La matriz de confusión permite ver con más detalle qué tipos de resultados el modelo predice bien y en cuáles se equivoca.

Cada fila representa el resultado real y cada columna representa la predicción del modelo. Los valores de la diagonal principal son aciertos; los demás valores son errores de clasificación.


In [ ]:
!pip install matplotlib 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_2022, preds_2022)

labels = ["Empate", "Victoria local", "Victoria visitante"]

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot()

plt.title("Matriz de confusión - Copa 2022")
plt.show()


#### Correlación entre variables

La correlación ayuda a explorar la relación lineal entre variables numéricas. En este caso la usamos como una herramienta de análisis rápido para entender si algunas variables se mueven junto con la variable objetivo.

Una correlación cercana a 1 indica una relación positiva fuerte; cercana a -1 indica una relación negativa fuerte; cercana a 0 indica poca relación lineal. Esta lectura es útil, pero no prueba causalidad ni garantiza que una variable sea buena por sí sola.


In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

features = [
    "elo_diff",
    "goals_diff",
    "defense_diff",
    "win_rate_diff",
    "match_result"   # Variable objetivo.
]

corr = df[features].corr()

plt.figure(figsize=(8,6))
sns.heatmap(
    corr,
    annot=True,       # Mostrar valores en cada celda.
    fmt=".2f",        # Mostrar dos decimales.
    cmap="coolwarm",  
    square=True
)

plt.title("Matriz de correlación")
plt.show()


#### Encapsulación del modelo para hacer predicciones

Ya entrenamos los modelos y calculamos las métricas necesarias. Ahora vamos a preparar una función que reciba dos equipos y devuelva una predicción fácil de interpretar.

La función calculará las mismas variables que usamos durante el entrenamiento y devolverá:
- Probabilidad de victoria del equipo local.
- Probabilidad de empate.
- Probabilidad de victoria del equipo visitante.
- Goles estimados para cada equipo.

Este paso es importante porque convierte el entrenamiento en una herramienta reutilizable para hacer consultas nuevas.


In [ ]:
ratings_global = ratings
team_stats_global = team_stats
model_global = models["rf"]
home_goals_model_global = home_goals_model
away_goals_model_global = away_goals_model

In [ ]:
import pandas as pd

def predict_match_simple(home_team, away_team):

    rating_home = ratings_global.get(home_team, 1500)
    rating_away = ratings_global.get(away_team, 1500)

    elo_diff = rating_home - rating_away

    # Si un equipo no está en el histórico, usamos valores promedio para permitir la predicción.
    teamA = team_stats_global.loc[home_team].to_dict() if home_team in team_stats_global.index else {
        "Goals_For": 1.2, "avg_goals_against": 1.2, "win_rate": 0.5
    }

    teamB = team_stats_global.loc[away_team].to_dict() if away_team in team_stats_global.index else {
        "Goals_For": 1.2, "avg_goals_against": 1.2, "win_rate": 0.5
    }

    features = pd.DataFrame([{
        "elo_diff": rating_home - rating_away,
        "goals_diff": teamA["Goals_For"] - teamB["Goals_For"],
        "defense_diff": teamA["avg_goals_against"] - teamB["avg_goals_against"],
        "win_rate_diff": teamA["win_rate"] - teamB["win_rate"]
    }])

    probs = model_global.predict_proba(features)[0]

    home_goals = home_goals_model_global.predict(features)[0]
    away_goals = away_goals_model_global.predict(features)[0]

    class_mapping = dict(zip(model_global.classes_, probs))

    return {
        "home_win": float(class_mapping.get(1, 0)),
        "draw": float(class_mapping.get(0, 0)),
        "away_win": float(class_mapping.get(2, 0)),
        "home_goals": float(home_goals),
        "away_goals": float(away_goals)
    }


#### Predicción de un partido entre dos equipos

En esta celda puedes definir el equipo local y el equipo visitante, ejecutar la función de predicción y revisar el resultado.

Para probar otros escenarios, cambia los valores de `home_team` y `away_team`. Usa los nombres tal como aparecen en los datos para obtener mejores resultados.


In [ ]:
home_team = "Brazil"
away_team = "Argentina"

result = predict_match_simple(home_team, away_team)

print(f"Victoria de {home_team}: {result['home_win']:.2%}")
print(f"Empate: {result['draw']:.2%}")
print(f"Victoria de {away_team}: {result['away_win']:.2%}")
print(f"Marcador estimado ({home_team} vs {away_team}): {result['home_goals']:.1f} x {result['away_goals']:.1f}")


#### Equipos disponibles para predicción

La siguiente celda muestra los equipos detectados en el conjunto de datos. Usa esta lista como referencia para parametrizar `home_team` y `away_team` en la predicción anterior.

Si un equipo no aparece en la lista, la función usará valores promedio por defecto. Esto permite que la función no falle, pero la predicción será menos representativa.


In [ ]:
from pyspark.sql.functions import col

home_teams = bronze_df.select(col("home_team_name").alias("team"))
away_teams = bronze_df.select(col("away_team_name").alias("team"))

teams = (
    home_teams
    .union(away_teams)
    .where(col("team").isNotNull())
    .dropDuplicates()
    .orderBy("team")
)

team_list = [r["team"] for r in teams.collect()]
max_items = 80

print(f"Equipos posibles ({len(team_list)}):")
body = "\n".join([f"- {t}" for t in team_list[:max_items]])
print(body)
if len(team_list) > max_items:
    print(f"- ... ({len(team_list) - max_items} equipos más)")

display(teams.limit(max_items))


#### Cierre y desafíos prácticos

En este notebook construimos un flujo completo de IA tradicional: cargamos datos, los limpiamos, creamos variables, entrenamos modelos, evaluamos resultados y generamos predicciones.

Para practicar y reforzar lo aprendido, prueba los siguientes desafíos:
- Cambia los equipos en la predicción y compara si los resultados parecen razonables.
- Agrega una nueva variable al modelo y revisa si mejora o empeora la exactitud.
- Prueba otro algoritmo de `scikit-learn` y compara su desempeño con Random Forest y Regresión Logística.
- Analiza la matriz de confusión e identifica qué tipo de resultado parece más difícil de predecir.

Recuerda: el objetivo principal es entender el flujo de trabajo. En un caso real, antes de llevar un modelo a producción se haría una validación más rigurosa y se revisarían posibles sesgos en los datos.
